# Time-series analysis

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-10-time-series.ipynb)

Time-series SQL comes down to three techniques: bucket timestamps into periods, fill the periods that have no data,
and compare each row with its neighbours in time. Most of the difficulty is at the edges: time zones, boundaries,
and days with no data at all.

This chapter covers period buckets, calendar spines, gaps and islands, sessions, cohorts and cumulative counts. It
ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Set the session time zone once, so timestamps mean the same moment on every machine.

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW events AS
SELECT * FROM VALUES
  (1,  1, TIMESTAMP '2023-01-01 09:00:00', 'login'),
  (2,  1, TIMESTAMP '2023-01-02 09:05:00', 'login'),
  (3,  1, TIMESTAMP '2023-01-03 08:55:00', 'login'),
  (4,  1, TIMESTAMP '2023-01-05 09:00:00', 'login'),
  (5,  1, TIMESTAMP '2023-01-06 09:10:00', 'login'),
  (6,  2, TIMESTAMP '2023-01-01 10:00:00', 'login'),
  (7,  2, TIMESTAMP '2023-01-02 10:00:00', 'login'),
  (8,  2, TIMESTAMP '2023-01-04 10:00:00', 'login'),
  (9,  3, TIMESTAMP '2023-02-01 10:00:00', 'view'),
  (10, 3, TIMESTAMP '2023-02-01 10:05:00', 'click'),
  (11, 3, TIMESTAMP '2023-02-01 10:25:00', 'view'),
  (12, 3, TIMESTAMP '2023-02-01 11:30:00', 'purchase'),
  (13, 3, TIMESTAMP '2023-02-01 11:45:00', 'view'),
  (14, 4, TIMESTAMP '2023-02-01 10:00:00', 'login'),
  (15, 4, TIMESTAMP '2023-02-01 10:20:00', 'view')
AS events(event_id, user_id, event_ts, event_type)
""")

### The example data

- `orders` has 14 orders from 6 customers, placed between 2023-01-02 and 2023-03-30. Order dates are `DATE` values.
  Most orders are `completed`; two are `cancelled` and one is `pending`.
- `events` has 15 events with `TIMESTAMP` values in UTC. Users 1 and 2 log in once a day in early January 2023, each
  with a missing day (user 1 skips 2023-01-04, user 2 skips 2023-01-03). Users 3 and 4 are clickstreams on
  2023-02-01, a few minutes apart, with one long pause for user 3.

The asserts below compare timestamps as strings (`date_format`). Python converts a collected `TIMESTAMP` into the
machine's local time zone, so a string is the only comparison that gives the same answer on every machine.

In [ ]:
assert spark.sql("SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM orders").collect() == [(14, 6)]
assert spark.sql("SELECT COUNT(*), COUNT(DISTINCT user_id) FROM events").collect() == [(15, 4)]

## 1. Time zones and boundaries

A `TIMESTAMP` in Spark stores an **instant**: one point on the global timeline. The same instant has a different
**wall-clock time** (the date and time printed on a local clock) in each time zone. Spark prints timestamps in the
session time zone, which the setup set to UTC.

The calendar day an event belongs to depends on the zone you choose. 2023-01-01 23:30 UTC is already 2023-01-02 08:30
in Tokyo (UTC+9) and still 2023-01-01 15:30 in Los Angeles (UTC-8 in winter). `from_utc_timestamp(ts, zone)` returns
the wall-clock time in `zone`, so casting it to `DATE` gives the local day.

Keep timestamps in UTC inside the warehouse, and convert only when a report needs local days.

In [ ]:
q = """
SELECT date_format(ts, 'yyyy-MM-dd HH:mm')                                        AS utc_time,
       CAST(ts AS DATE)                                                           AS utc_day,
       date_format(from_utc_timestamp(ts, 'Asia/Tokyo'), 'yyyy-MM-dd HH:mm')          AS tokyo_time,
       CAST(from_utc_timestamp(ts, 'Asia/Tokyo') AS DATE)                         AS tokyo_day,
       CAST(from_utc_timestamp(ts, 'America/Los_Angeles') AS DATE)                AS la_day
FROM VALUES (TIMESTAMP '2023-01-01 23:30:00') AS t(ts)
"""
spark.sql(q).show()

from datetime import date

assert spark.sql(q).collect() == [
    ("2023-01-01 23:30", date(2023, 1, 1), "2023-01-02 08:30", date(2023, 1, 2), date(2023, 1, 1))
]

### Half-open intervals

A time range written as `[start, end)` is **half-open**: it includes `start` and excludes `end`. Consecutive
half-open ranges meet without overlapping, so every instant belongs to exactly one of them.

`BETWEEN a AND b` is closed on both ends. Worked by hand with three events and two daily ranges:

| event | `BETWEEN` day 1 | `BETWEEN` day 2 | `[start, end)` day 1 | `[start, end)` day 2 |
|---|---|---|---|---|
| 2023-02-01 23:59:59 | yes | no | yes | no |
| 2023-02-02 00:00:00 | **yes** | **yes** | no | yes |
| 2023-02-02 12:00:00 | no | yes | no | yes |

With `BETWEEN`, the event at midnight is counted on both days, so the daily totals add up to 4 for 3 events.
With half-open ranges they add up to 3.

In [ ]:
q = """
WITH ev AS (
  SELECT * FROM VALUES
    (TIMESTAMP '2023-02-01 23:59:59'),
    (TIMESTAMP '2023-02-02 00:00:00'),
    (TIMESTAMP '2023-02-02 12:00:00')
  AS ev(ts)
)
SELECT
  SUM(CASE WHEN ts BETWEEN TIMESTAMP '2023-02-01 00:00:00' AND TIMESTAMP '2023-02-02 00:00:00' THEN 1 ELSE 0 END)
    AS closed_day1,
  SUM(CASE WHEN ts BETWEEN TIMESTAMP '2023-02-02 00:00:00' AND TIMESTAMP '2023-02-03 00:00:00' THEN 1 ELSE 0 END)
    AS closed_day2,
  SUM(CASE WHEN ts >= TIMESTAMP '2023-02-01 00:00:00' AND ts < TIMESTAMP '2023-02-02 00:00:00' THEN 1 ELSE 0 END)
    AS half_open_day1,
  SUM(CASE WHEN ts >= TIMESTAMP '2023-02-02 00:00:00' AND ts < TIMESTAMP '2023-02-03 00:00:00' THEN 1 ELSE 0 END)
    AS half_open_day2
FROM ev
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 2, 1, 2)]

### Where you'll use it

- **Global reporting:** an order placed at 23:30 UTC belongs to the next day in Tokyo. Every daily report must state
  which zone defines its days.
- **Billing and SLA periods:** invoices, monthly usage and uptime windows are half-open ranges, so a usage record at
  exactly midnight is billed once.
- **Daylight saving:** a local day can be 23 or 25 hours long. Converting from UTC with a named zone such as
  `America/Los_Angeles` handles this; adding a fixed offset does not.

## 2. Bucketing into periods

To **bucket** a timestamp is to replace it with the start of the period that contains it. All rows in the same
period then share one value, and `GROUP BY` that value gives one row per period.

Spark has two functions for this, with the arguments in opposite orders:

- `trunc(date, 'MM')` returns a `DATE`. Units include `'YEAR'`, `'QUARTER'`, `'MM'` (month) and `'WEEK'`.
- `date_trunc('MONTH', ts)` returns a `TIMESTAMP`. It also accepts `'DAY'`, `'HOUR'` and `'MINUTE'`, so use it for
  periods shorter than a day.

Weeks start on Monday. Worked by hand:

| date | weekday | `'MM'` | `'WEEK'` | `'QUARTER'` |
|---|---|---|---|---|
| 2023-01-05 | Thursday | 2023-01-01 | 2023-01-02 | 2023-01-01 |
| 2023-02-15 | Wednesday | 2023-02-01 | 2023-02-13 | 2023-01-01 |

In [ ]:
q = """
SELECT d, trunc(d, 'MM') AS month_start, trunc(d, 'WEEK') AS week_start, trunc(d, 'QUARTER') AS quarter_start
FROM VALUES (DATE '2023-01-05'), (DATE '2023-02-15') AS t(d)
ORDER BY d
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (date(2023, 1, 5), date(2023, 1, 1), date(2023, 1, 2), date(2023, 1, 1)),
    (date(2023, 2, 15), date(2023, 2, 1), date(2023, 2, 13), date(2023, 1, 1)),
]

Bucket `orders` by month and count. January has orders 1, 4, 7 and 11; February and March have five each.

In [ ]:
q = """
SELECT trunc(order_date, 'MM') AS month_start, COUNT(*) AS orders
FROM orders
GROUP BY trunc(order_date, 'MM')
ORDER BY month_start
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(date(2023, 1, 1), 4), (date(2023, 2, 1), 5), (date(2023, 3, 1), 5)]

For hourly buckets use `date_trunc('HOUR', ts)`. On 2023-02-01, events 9 to 11, 14 and 15 fall in the 10:00 hour
and events 12 and 13 in the 11:00 hour.

In [ ]:
q = """
SELECT date_format(date_trunc('HOUR', event_ts), 'yyyy-MM-dd HH:mm') AS hour_start, COUNT(*) AS events
FROM events
WHERE event_ts >= TIMESTAMP '2023-02-01 00:00:00' AND event_ts < TIMESTAMP '2023-02-02 00:00:00'
GROUP BY date_trunc('HOUR', event_ts)
ORDER BY hour_start
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("2023-02-01 10:00", 5), ("2023-02-01 11:00", 2)]

### Where you'll use it

- **Daily, weekly and monthly metrics:** every time-series dashboard is a bucket expression in `GROUP BY`.
- **Week-over-week comparisons:** `trunc(d, 'WEEK')` gives a stable Monday key, so this week and last week line up.
- **Traffic monitoring:** `date_trunc('HOUR', ts)` or `'MINUTE'` buckets requests to find peaks.

## 3. Filling gaps with a calendar spine

`GROUP BY` returns one row per group that **exists** in the data. A day with no logins has no rows, so it has no group,
and a chart built from the result shows a hole instead of a zero.

A **calendar spine** is a table with exactly one row per period in the range you report on, whether or not anything
happened. Build it first, then `LEFT JOIN` the activity onto it, so every spine row survives. Missing counts arrive as
NULL; `COALESCE(x, 0)` turns them into zeros.

`sequence(start, stop, step)` returns an array of every value from `start` to `stop`, both included. `explode` turns
an array into one row per element.

Report on 2023-01-01 to 2023-01-07. The logins in that week fall on six days; there are none on 2023-01-07, so
`GROUP BY` alone returns six rows.

In [ ]:
daily_only = spark.sql("""
SELECT CAST(event_ts AS DATE) AS day, COUNT(*) AS logins
FROM events
WHERE event_type = 'login'
  AND event_ts >= TIMESTAMP '2023-01-01 00:00:00' AND event_ts < TIMESTAMP '2023-01-08 00:00:00'
GROUP BY CAST(event_ts AS DATE)
""")
assert daily_only.count() == 6

With the spine on the left of the join, 2023-01-07 appears with zero logins. Filter the activity (`event_type =
'login'`) before the join, inside `daily`. A filter on the right-hand table in the outer `WHERE` would remove the
spine rows that found no match. The join also drops activity outside the spine, such as user 4's login on
2023-02-01.

In [ ]:
q = """
WITH spine AS (
  SELECT explode(sequence(DATE '2023-01-01', DATE '2023-01-07', INTERVAL 1 DAY)) AS day
),
daily AS (
  SELECT CAST(event_ts AS DATE) AS day, COUNT(*) AS logins
  FROM events
  WHERE event_type = 'login'
  GROUP BY CAST(event_ts AS DATE)
)
SELECT s.day, COALESCE(d.logins, 0) AS logins
FROM spine s
LEFT JOIN daily d ON d.day = s.day
ORDER BY s.day
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (date(2023, 1, 1), 2), (date(2023, 1, 2), 2), (date(2023, 1, 3), 1), (date(2023, 1, 4), 1),
    (date(2023, 1, 5), 1), (date(2023, 1, 6), 1), (date(2023, 1, 7), 0),
]

### Where you'll use it

- **Continuous charts:** a day with no sales must show zero, not disappear. A calendar table left-joined to the facts
  does this.
- **Per-entity grids:** `CROSS JOIN` a spine of days with a list of users or stores to get one row per entity per
  day, then left-join activity to find who was missing on which day.
- **Moving averages:** a 7-day average over a series with missing days averages the wrong rows. Fill the spine first.

## 4. Gaps and islands

An **island** is a maximal run of consecutive days with activity. A **gap** is the run of missing days between two
islands. The task is to label each row with the island it belongs to.

Number each user's active days in date order with `ROW_NUMBER`, and subtract that number from the date. Worked by hand
for user 1, who was active on January 1, 2, 3, 5 and 6:

| day | row number | day minus row number |
|---|---|---|
| 2023-01-01 | 1 | 2022-12-31 |
| 2023-01-02 | 2 | 2022-12-31 |
| 2023-01-03 | 3 | 2022-12-31 |
| 2023-01-05 | 4 | 2023-01-01 |
| 2023-01-06 | 5 | 2023-01-01 |

Inside a run, the day and the row number both go up by 1 on each row, so their difference stays constant. Across a gap
of `k` missing days, the day jumps by `k + 1` but the row number by only 1, so the difference moves forward by `k`.
The difference is therefore a label for the island, and `GROUP BY user_id, island` gives one row per island.

`date_sub(d, n)` subtracts `n` days. `n` must be an `INT`, and `ROW_NUMBER` returns a `BIGINT`, so cast it.

In [ ]:
q = """
WITH daily AS (
  SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
),
grouped AS (
  SELECT user_id, d,
         date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
  FROM daily
)
SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS days
FROM grouped
GROUP BY user_id, island
ORDER BY user_id, start_day
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, date(2023, 1, 1), date(2023, 1, 3), 3),
    (1, date(2023, 1, 5), date(2023, 1, 6), 2),
    (2, date(2023, 1, 1), date(2023, 1, 2), 2),
    (2, date(2023, 1, 4), date(2023, 1, 4), 1),
    (3, date(2023, 2, 1), date(2023, 2, 1), 1),
    (4, date(2023, 2, 1), date(2023, 2, 1), 1),
]

### Why deduplicate first

The trick assumes one row per user per day. User 3 has five events on 2023-02-01. Without `DISTINCT`, those five rows
get row numbers 1 to 5 on the same date, so the differences are all different and one day splits into five islands.

In [ ]:
q = """
WITH grouped AS (
  SELECT user_id,
         date_sub(CAST(event_ts AS DATE),
                  CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS INT)) AS island
  FROM events
  WHERE user_id = 3
)
SELECT COUNT(DISTINCT island) AS islands FROM grouped
"""
assert spark.sql(q).collect() == [(5,)]

### Where you'll use it

- **Streaks:** consecutive days a user logged in, for streak badges and engagement metrics.
- **Outages:** consecutive minutes a health check failed, merged into one incident with a start and an end.
- **Subscriptions:** consecutive paid months merged into one continuous subscription period.

## 5. Sessions

A **session** is a burst of activity by one user. The usual rule: a new session starts when the gap since the same
user's previous event is more than 30 minutes. The threshold and the choice of `>` over `>=` are business rules.

The query has three steps:

1. `LAG(event_ts)` gives the previous event's timestamp within the same user (`PARTITION BY user_id`). The first event
   of each user has no previous event, so `LAG` returns NULL.
2. Flag each event with 1 if it starts a session (no previous event, or a gap over 30 minutes), else 0.
   `unix_timestamp` converts a timestamp to seconds, so the gap in minutes is the difference divided by 60.
3. A running `SUM` of the flags numbers the sessions. With `ORDER BY` and no frame, the window covers every earlier row
   up to the current one, so the sum counts the session starts seen so far.

`event_id` is a tie-breaker in the ordering, so two events at the same instant still have a fixed order.

Worked by hand for user 3:

| event_ts | previous | gap (min) | starts_session | session_no |
|---|---|---|---|---|
| 10:00 | NULL | | 1 | 1 |
| 10:05 | 10:00 | 5 | 0 | 1 |
| 10:25 | 10:05 | 20 | 0 | 1 |
| 11:30 | 10:25 | 65 | 1 | 2 |
| 11:45 | 11:30 | 15 | 0 | 2 |

In [ ]:
q = """
WITH ordered AS (
  SELECT user_id, event_ts, event_id,
         LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
  FROM events
),
flagged AS (
  SELECT user_id, event_ts, event_id,
         CASE WHEN prev_ts IS NULL
                OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
              THEN 1 ELSE 0 END AS starts_session
  FROM ordered
)
SELECT user_id, date_format(event_ts, 'HH:mm') AS event_time,
       SUM(starts_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_no
FROM flagged
WHERE user_id IN (3, 4)
ORDER BY user_id, event_ts, event_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (3, "10:00", 1), (3, "10:05", 1), (3, "10:25", 1), (3, "11:30", 2), (3, "11:45", 2),
    (4, "10:00", 1), (4, "10:20", 1),
]

### Summarising sessions

Once every event has a `session_no`, group by `user_id, session_no` to get one row per session: its start, its end,
its event count and its length. User 3's first session runs from 10:00 to 10:25 (25 minutes, 3 events).

In [ ]:
q = """
WITH ordered AS (
  SELECT user_id, event_ts, event_id,
         LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
  FROM events
),
flagged AS (
  SELECT user_id, event_ts, event_id,
         CASE WHEN prev_ts IS NULL
                OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
              THEN 1 ELSE 0 END AS starts_session
  FROM ordered
),
numbered AS (
  SELECT user_id, event_ts,
         SUM(starts_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_no
  FROM flagged
)
SELECT user_id, session_no,
       date_format(MIN(event_ts), 'HH:mm') AS session_start,
       date_format(MAX(event_ts), 'HH:mm') AS session_end,
       COUNT(*) AS events,
       (unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts))) DIV 60 AS minutes
FROM numbered
WHERE user_id IN (3, 4)
GROUP BY user_id, session_no
ORDER BY user_id, session_no
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (3, 1, "10:00", "10:25", 3, 25),
    (3, 2, "11:30", "11:45", 2, 15),
    (4, 1, "10:00", "10:20", 2, 20),
]

### Where you'll use it

- **Web analytics:** Google Analytics and product analytics tools end a session after 30 minutes without activity.
- **Funnels:** "did the user add to cart and buy within the same session?" needs a session id on every event.
- **Machine and IoT data:** runs of sensor readings separated by idle periods become jobs or trips.

## 6. Cohorts and retention

A **cohort** is the group of customers who first appeared in the same period. Here the period is a month, and a
customer's **cohort month** is the month of their first order. A customer is **active** in a month if they placed any
order in it (this chapter counts every status; a real report might count only completed orders).

Worked by hand from `orders`:

| customer | months with orders | cohort month |
|---|---|---|
| 1 | Jan, Feb, Mar | Jan |
| 2 | Jan, Feb, Mar | Jan |
| 3 | Jan, Feb | Jan |
| 4 | Feb, Mar | Feb |
| 5 | Jan, Mar | Jan |
| 6 | Feb, Mar | Feb |

The January cohort has 4 customers (1, 2, 3, 5). In February 3 of them are active (1, 2, 3), and in March 3 (1, 2, 5).
The February cohort has 2 customers (4, 6), and both are active in March.

`activity` keeps one row per customer per month (`DISTINCT`), so a customer with two orders in one month counts once.

In [ ]:
q = """
WITH first_order AS (
  SELECT customer_id, MIN(trunc(order_date, 'MM')) AS cohort_month
  FROM orders
  GROUP BY customer_id
),
activity AS (
  SELECT DISTINCT customer_id, trunc(order_date, 'MM') AS activity_month
  FROM orders
)
SELECT f.cohort_month, a.activity_month, COUNT(*) AS active_customers
FROM first_order f
JOIN activity a ON a.customer_id = f.customer_id
GROUP BY f.cohort_month, a.activity_month
ORDER BY f.cohort_month, a.activity_month
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (date(2023, 1, 1), date(2023, 1, 1), 4),
    (date(2023, 1, 1), date(2023, 2, 1), 3),
    (date(2023, 1, 1), date(2023, 3, 1), 3),
    (date(2023, 2, 1), date(2023, 2, 1), 2),
    (date(2023, 2, 1), date(2023, 3, 1), 2),
]

### Retention rate by month offset

Cohorts start in different months, so compare them by **month offset**: the number of months since the cohort month
(0 for the first month). `months_between` of two first-of-month dates is a whole number. The **retention rate** is the
number of active customers divided by the cohort size. For the January cohort that is 4/4, 3/4 and 3/4.

`/` on two integers returns a `DOUBLE` in Spark, so `3 / 4` is `0.75`.

In [ ]:
q = """
WITH first_order AS (
  SELECT customer_id, MIN(trunc(order_date, 'MM')) AS cohort_month
  FROM orders
  GROUP BY customer_id
),
cohort_size AS (
  SELECT cohort_month, COUNT(*) AS size FROM first_order GROUP BY cohort_month
),
activity AS (
  SELECT DISTINCT customer_id, trunc(order_date, 'MM') AS activity_month
  FROM orders
),
active AS (
  SELECT f.cohort_month,
         CAST(months_between(a.activity_month, f.cohort_month) AS INT) AS month_offset,
         COUNT(*) AS active_customers
  FROM first_order f
  JOIN activity a ON a.customer_id = f.customer_id
  GROUP BY f.cohort_month, CAST(months_between(a.activity_month, f.cohort_month) AS INT)
)
SELECT a.cohort_month, a.month_offset, a.active_customers / s.size AS retention
FROM active a
JOIN cohort_size s ON s.cohort_month = a.cohort_month
ORDER BY a.cohort_month, a.month_offset
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (date(2023, 1, 1), 0, 1.0), (date(2023, 1, 1), 1, 0.75), (date(2023, 1, 1), 2, 0.75),
    (date(2023, 2, 1), 0, 1.0), (date(2023, 2, 1), 1, 1.0),
]

### Where you'll use it

- **Retention curves:** the share of users who signed up in January and came back in month 1, 2, 3.
- **Marketing attribution:** compare cohorts acquired by different campaigns at the same month offset.
- **Subscription businesses:** revenue retention per cohort is a standard board metric.

## 7. Cumulative counts

"How many distinct customers had ordered by this date?" is a **running distinct count**. The obvious window,
`COUNT(DISTINCT customer_id) OVER (ORDER BY order_date)`, is rejected by Spark with
`DISTINCT_WINDOW_FUNCTION_UNSUPPORTED`. A correlated subquery that recounts all history for every date works but
rescans the table once per date.

The way round it: a customer becomes "new" on exactly one date, their first order. So the number of distinct customers
up to date `D` equals the number of customers whose first order is on or before `D`. Find each first order once, count
new customers per date, and take a running `SUM` of those counts.

First orders, worked by hand: customer 5 on 2023-01-02, 1 on 01-05, 2 on 01-20, 3 on 01-25, 4 on 02-14 and 6 on
02-18. Each date has one new customer, so the running total goes 1, 2, 3, 4, 5, 6.

`SUM(COUNT(*)) OVER (...)` is legal because window functions run after `GROUP BY`: the inner `COUNT(*)` is the
group's count, and the outer `SUM` adds those counts across rows.

In [ ]:
q = """
WITH first_order AS (
  SELECT customer_id, MIN(order_date) AS first_date
  FROM orders
  GROUP BY customer_id
)
SELECT first_date,
       COUNT(*) AS new_customers,
       SUM(COUNT(*)) OVER (ORDER BY first_date) AS cumulative_customers
FROM first_order
GROUP BY first_date
ORDER BY first_date
"""
spark.sql(q).show()
running = spark.sql(q).collect()
assert running == [
    (date(2023, 1, 2), 1, 1), (date(2023, 1, 5), 1, 2), (date(2023, 1, 20), 1, 3),
    (date(2023, 1, 25), 1, 4), (date(2023, 2, 14), 1, 5), (date(2023, 2, 18), 1, 6),
]

To check the shortcut, compute the same numbers the slow way: for each of those dates, count the distinct customers
with any order on or before it. The two methods agree.

In [ ]:
brute_force = spark.sql("""
WITH dates AS (SELECT DISTINCT MIN(order_date) OVER (PARTITION BY customer_id) AS d FROM orders)
SELECT dates.d, COUNT(DISTINCT o.customer_id) AS customers_so_far
FROM dates
JOIN orders o ON o.order_date <= dates.d
GROUP BY dates.d
ORDER BY dates.d
""").collect()
assert [(r.first_date, r.cumulative_customers) for r in running] == [tuple(r) for r in brute_force]

### Where you'll use it

- **Running distinct counts:** unique customers to date, computed without a full rescan each day.
- **Growth charts:** cumulative sign-ups or activated accounts over time.
- **Incremental pipelines:** store each entity's first-seen date once; the running total then needs only new rows.

## Interview notes

- **Late and out-of-order data.** Events arrive after their period has been reported. Reprocess the affected
  periods, or accept a short window of lateness and say so.
- **Define the session.** "A new session after 30 minutes" is a business rule. Ask whether the gap is measured
  between consecutive events, whether exactly 30 minutes starts a new session, and whether a session can span midnight.
- **Gaps and islands.** Know the "date minus row number" trick, and say that it needs one row per entity per day,
  so deduplicate first.
- **Time zones.** Store timestamps in UTC and convert for display. Daylight saving changes make local-time day
  lengths differ, which breaks naive "24 hours = one day" arithmetic.
- **Missing periods.** Mention the calendar spine whenever a time series feeds a chart or a moving average.

## Summary

- Store timestamps in UTC, convert with a named zone for local days, and use half-open `[start, end)` ranges.
- Bucket with `trunc` (returns a `DATE`) or `date_trunc` (returns a `TIMESTAMP`, and supports hours and minutes).
- Build a calendar spine and left-join activity onto it, so empty periods show as zero.
- Gaps and islands: subtract the row number from the date to label each run of consecutive days.
- Sessions: compare each event with the previous one using `LAG`, flag the starts, and take a running sum.
- Cohorts group customers by their first period; retention divides active customers by cohort size.
- For running distinct counts, count first appearances and take a running sum.

## Practice questions

1. For each month, how many orders were placed, and what was the total amount of completed orders? Sort by month.
2. For users 1 and 2 and the days 2023-01-01 to 2023-01-06, on how many days did each user not log in? Sort by
   `user_id`.
3. How many sessions does each user have, using the 30-minute rule? Sort by `user_id`.
4. What is each user's longest streak of consecutive active days? Sort by `user_id`.
5. What share of the January 2023 cohort was active in February 2023, and what share in March 2023? Sort by month.
6. How many distinct customers had ordered by the end of each month from January to March 2023? Include a month with
   no new customers. Sort by month.

## Answers

### Question 1

Bucket with `trunc(order_date, 'MM')`. Count every row, and sum the amount only when the status is `completed`.

In [ ]:
q = """
SELECT trunc(order_date, 'MM') AS month_start,
       COUNT(*) AS orders,
       SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed_amount
FROM orders
GROUP BY trunc(order_date, 'MM')
ORDER BY month_start
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (date(2023, 1, 1), 4, 1020), (date(2023, 2, 1), 5, 655), (date(2023, 3, 1), 5, 1320),
]

### Question 2

`CROSS JOIN` a spine of days with the two users to get every user-day pair, then left-join the distinct login days and
count the pairs that found no match. User 1 missed 2023-01-04; user 2 missed 01-03, 01-05 and 01-06.

In [ ]:
q = """
WITH spine AS (
  SELECT explode(sequence(DATE '2023-01-01', DATE '2023-01-06', INTERVAL 1 DAY)) AS day
),
users AS (
  SELECT * FROM VALUES (1), (2) AS users(user_id)
),
logins AS (
  SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS day
  FROM events
  WHERE event_type = 'login'
)
SELECT u.user_id, SUM(CASE WHEN l.day IS NULL THEN 1 ELSE 0 END) AS missed_days
FROM users u
CROSS JOIN spine s
LEFT JOIN logins l ON l.user_id = u.user_id AND l.day = s.day
GROUP BY u.user_id
ORDER BY u.user_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 1), (2, 3)]

### Question 3

Number the sessions as in section 5. The highest session number per user is the session count.

In [ ]:
q = """
WITH ordered AS (
  SELECT user_id, event_ts, event_id,
         LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
  FROM events
),
flagged AS (
  SELECT user_id, event_ts, event_id,
         CASE WHEN prev_ts IS NULL
                OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
              THEN 1 ELSE 0 END AS starts_session
  FROM ordered
)
SELECT user_id, MAX(session_no) AS sessions
FROM (
  SELECT user_id, SUM(starts_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_no
  FROM flagged
)
GROUP BY user_id
ORDER BY user_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 5), (2, 3), (3, 2), (4, 1)]

### Question 4

Find the islands as in section 4, then take the largest island size per user.

In [ ]:
q = """
WITH daily AS (
  SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
),
grouped AS (
  SELECT user_id,
         date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
  FROM daily
),
islands AS (
  SELECT user_id, island, COUNT(*) AS days FROM grouped GROUP BY user_id, island
)
SELECT user_id, MAX(days) AS longest_streak
FROM islands
GROUP BY user_id
ORDER BY user_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 3), (2, 2), (3, 1), (4, 1)]

### Question 5

Keep only customers whose cohort month is January, count the distinct ones active in each later month, and divide by
the cohort size (4).

In [ ]:
q = """
WITH first_order AS (
  SELECT customer_id, MIN(trunc(order_date, 'MM')) AS cohort_month
  FROM orders
  GROUP BY customer_id
),
jan AS (
  SELECT customer_id FROM first_order WHERE cohort_month = DATE '2023-01-01'
)
SELECT trunc(o.order_date, 'MM') AS activity_month,
       COUNT(DISTINCT o.customer_id) / (SELECT COUNT(*) FROM jan) AS share_active
FROM orders o
JOIN jan j ON j.customer_id = o.customer_id
WHERE o.order_date >= DATE '2023-02-01' AND o.order_date < DATE '2023-04-01'
GROUP BY trunc(o.order_date, 'MM')
ORDER BY activity_month
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(date(2023, 2, 1), 0.75), (date(2023, 3, 1), 0.75)]

### Question 6

Count new customers per month from each customer's first order, put a month spine on the left so March appears with
zero new customers, and take a running sum.

In [ ]:
q = """
WITH spine AS (
  SELECT explode(sequence(DATE '2023-01-01', DATE '2023-03-01', INTERVAL 1 MONTH)) AS month_start
),
new_per_month AS (
  SELECT trunc(MIN(order_date), 'MM') AS month_start, customer_id
  FROM orders
  GROUP BY customer_id
),
counts AS (
  SELECT s.month_start, COUNT(n.customer_id) AS new_customers
  FROM spine s
  LEFT JOIN new_per_month n ON n.month_start = s.month_start
  GROUP BY s.month_start
)
SELECT month_start, new_customers,
       SUM(new_customers) OVER (ORDER BY month_start) AS customers_to_date
FROM counts
ORDER BY month_start
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (date(2023, 1, 1), 4, 4), (date(2023, 2, 1), 2, 6), (date(2023, 3, 1), 0, 6),
]